In [1]:
import os
import math
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn.functional as F  
from torch.utils.tensorboard import SummaryWriter

from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADVisualizer   import CADVisualizer
from HDVClassNet.PP_net import PPNet
from HDVClassNet.VoronoiDecorder import VoronoiDecoder
from Training.MainTrain import TrainingConfig, NN_Trainer
from neuraltomo_fem import run_fem_loss
from problems.ThickenShell import ThickenShell

import pyvista as pv


# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
# -------- PYVISTA BACKEND --------
def setup_pyvista(device):
    is_mac = sys.platform == "darwin"

    # Mac + MPS: prefer static to avoid VTK/trame hangs
    if is_mac :
        pv.OFF_SCREEN = True
        pv.set_jupyter_backend("static")
        backend = "static"
    else:
        try:
            pv.set_jupyter_backend("trame")
            backend = "trame"
        except Exception:
            pv.OFF_SCREEN = True
            pv.set_jupyter_backend("static")
            backend = "static"

    print(f"PyVista backend: {backend}")

setup_pyvista(device)


Code Directory: /home/newuser/VoronoiImp-main-1
Test Step files Directory: /home/newuser/VoronoiImp-main-1/Testparts
device: cuda
PyVista backend: trame


In [2]:
viz = CADVisualizer()
# Laoding model and extracting mesh and tensors as input
FreeFormSurf1  = TesPartsDir / "FreeFormCrv1.stp"
FreeFormSurf2A = TesPartsDir / "FreeFormSurf2A.STEP"
YachtBodypart  = TesPartsDir / "YachtBodypart.stp"
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Cube           = TesPartsDir / "Cube.stp"
CircularSur2   = TesPartsDir / "CircularSur2.stp"
Conic          = TesPartsDir / "Conic.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
FullCylinder   = TesPartsDir / "FullCylinder.stp"
Sphere         = TesPartsDir / "Sphere.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Tidebottle     = TesPartsDir / "Tidebottle.STEP"


shape_path = CircularSurf1

Case_name = shape_path.stem
print(Case_name)
generator = CADTensorGenerator(
    deflection=0.001,
    angle=0.001,
    metric_tol=1e-9,
    det_min=1e-5,
    n_u=50,
    n_v=50,
    device=device,
)

mesh_df, faces_df, tensors = generator.generate_from_file(
    shape_path=str(shape_path),
    input_ring=1,
    mode="mesh", #"1: mesh" "2:Sampled_points "
    M_per_face=2000,
    pool_size_factor=10,
    fps_pool_factor=4,
    use_fps=True,
    triangulation_max_edge_rel=0.1,
)

uv = tensors["uv"]
points_xyz = tensors["points_xyz"]
uv = tensors["uv"]
Xu = tensors["Xu"]
Xv = tensors["Xv"]
points_xyz = tensors["points_xyz"]
face_areas = tensors["face_areas"]
faces_ijk = tensors["faces_ijk"]
face_id = tensors["face_id"]
boundary_idx_ring1 = tensors["boundary_idx_ring1"]
pv_faces = tensors["pv_faces"]
bbx_all = list(tensors["BBX"].values())



xmin = min(b["xmin"] for b in bbx_all)
xmax = max(b["xmax"] for b in bbx_all)
ymin = min(b["ymin"] for b in bbx_all)
ymax = max(b["ymax"] for b in bbx_all)
zmin = min(b["zmin"] for b in bbx_all)
zmax = max(b["zmax"] for b in bbx_all)

dx = xmax - xmin
dy = ymax - ymin
dz = zmax - zmin


print(f"Number of faces: {tensors['num_faces']}")
print(f"Number of Sampled points: {uv.shape[0]}")
print(f"Global BBX dimensions: dx={dx:.4f}, dy={dy:.4f}, dz={dz:.4f}")

viz.visualize_show_Model(points_xyz, pv_faces)


pts = points_xyz.detach().cpu().numpy()
cloud = pv.PolyData(pts)
plotter = pv.Plotter()
plotter.add_mesh(cloud, render_points_as_spheres=True, point_size=6)
plotter.show()

CircularSurf1
MinVolFrac: 0.11639995872974396
Number of faces: 1
Number of Sampled points: 2601
Global BBX dimensions: dx=10.0000, dy=2.2278, dz=3.9706


Widget(value='<iframe src="http://localhost:35397/index.html?ui=P_0x78c5001b5060_0&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:35397/index.html?ui=P_0x78c5001b4fd0_1&reconnect=auto" class="pyvi…

In [3]:
fixed_height_shell= 0.2
# shell_problem = ThickenShell(
#     thickness=fixed_height_shell,
#     BC_dir = "y",
#     Load_magnitude=0.0001,
#     voxel_size=0.1,
#     extra_layers=1,
#     tensors=tensors,
#     tangential_tol=0.1,
# )
shell_problem =None
fem =None
# fem = run_fem_loss.NeuralTOMOFEM(shell_problem, device=device, isotropic=False)
# shell_problem.debug_voxel_stats()
# shell_problem.show_voxels_surface_and_bc()

In [4]:
cfg = TrainingConfig(
    # ============================================================
    # Seed / geometry setup
    # Controls how many Voronoi generators exist and basic width bounds.
    # More seeds = richer structure but harder optimization.
    # ============================================================
    seed_number=5,
    use_anisotropy=False,         # False = isotropic metric; simpler, more stable
    fixed_height=fixed_height_shell,
    freeze_w=False,               # False = allow strut width to learn
    w_min=0.0005,                 # minimum allowed decoded strut width
    w_max=0.5,                    # maximum allowed decoded strut width

    # ============================================================
    # Main geometric / physical target
    # target_volfrac is the desired material usage after decoding.
    # ============================================================
    target_volfrac=0.2,

    # ============================================================
    # Geometric regularization terms
    # These prevent seeds from collapsing together or sticking to boundaries.
    # ============================================================
    seed_repulsion_sigma=0.08,    # interaction radius for seed-seed repulsion
    boundary_margin=0.05,         # how strongly seeds are pushed from open boundaries

    # ============================================================
    # Core loss weights
    # lam_vol   : pushes density toward target_volfrac
    # lam_rep   : spreads seeds apart
    # lam_bnd   : keeps seeds off boundaries
    # lam_strut : encourages strut-like density relative to edge field
    # lam_fem   : structural mechanics term (off here)
    # ============================================================
    lam_fem=0.0,
    lam_vol=20.0,
    lam_rep=0.015,
    lam_bnd=0.0,
    lam_strut=0.0005,
    lam_strut_edge=0.0,           # edge-preservation part of strut loss
    lam_strut_void=0.2,           # void-cleaning part of strut loss

    # ============================================================
    # Gate losses
    # use_gating           : enables per-seed activity gates
    # lam_gate             : keeps mean gate near gate_target_mean
    # lam_gate_binary      : pushes gates away from middle values toward 0 or 1
    # gate_target_mean     : desired average gate level across seeds
    # gate_warmup_steps    : gradually turns on mean-target gate loss
    # gate_binary_warmup_steps : gradually turns on binary gate sharpening loss
    # gate_sharpen_gamma   : makes effective structural gate more binary in decoder
    # gate_active_threshold: threshold only for logging / visualization
    # ============================================================

    
    use_gating=False,
    lam_gate_count = 0.005,
    lam_gate_binary=0.1,
    gate_target_count = 10.0,
    gate_warmup_steps=20,
    gate_binary_warmup_steps=80,
    gate_sharpen_gamma=2.0,
    gate_active_threshold=0.3,
    gate_eps=1e-8,                # numerical stability for log(gate)
    gate_bias_init=2.0,           # initial sigmoid gate ~0.88, so seeds start active

    # ============================================================
    # Learning rates
    # seed_refine usually needs larger LR than MLP heads.
    # gate head is intentionally smaller to avoid instant collapse.
    # ============================================================
    lr_seed_refine=0.003,
    lr_delta_head=1e-4,
    lr_mlp=2e-4,
    lr_w_head=2e-3,
    lr_h_head=2e-4,
    lr_gate_head=5e-5,

    # ============================================================
    # FEM / normalization
    # FEM is off here, so these mostly stay inactive in this run.
    # normalize_losses=False means raw loss magnitudes are used directly.
    # ============================================================
    comp_normalize_by=None,
    normalize_losses=False,
    fem_density_floor=0.02,
    skip_bad_fem_steps=True,

    # ============================================================
    # Optimization schedule
    # num_steps          : total training iterations
    # log_every          : console logging frequency
    # early stop params  : unused here because num_steps < early_stop_start
    # ============================================================
    num_steps=5000,
    context_vector_size=8,
    log_every=100,
    early_stop_start=500,
    patience=500,
    min_delta=1e-10,

    # ============================================================
    # Decoder softness / geometry shaping
    # tau  : softmax temperature in Voronoi competition
    #        smaller = sharper cell competition
    # beta : band sharpness / structural blending parameter in decoder
    # ============================================================
    tau=0.1,
    beta=0.02,

    # ============================================================
    # Scheduler / training stability
    # milestones are beyond this short run, so scheduler will not trigger here.
    # Offset_scale controls seed motion scale predicted by the network.
    # grad clipping prevents unstable updates.
    # ============================================================
    scheduler_milestones=(2000, 4000),
    scheduler_gamma=0.2,
    Offset_scale=5,
    save_fem_debug_history=True,
    grad_clip_norm=1.0,

    # ============================================================
    # TensorBoard logging
    # ============================================================
    experiment_name=str(Case_name),
    tensorboard_enabled=True,
    tb_log_histograms_every=100,

    # ============================================================
    # Timelapse / visualization
    # Higher UV resolution = nicer video but slower rendering.
    # ============================================================
    MakeTimelaps=True,
    timelapse_frame_step=10,
    TM_laps_res_u=150,
    TM_laps_res_v=150,
    TM_laps_Thr=0.25,
)

trainer = NN_Trainer(
    generator=generator,
    viz=viz,
    decoder_cls=VoronoiDecoder,
    ppnet_cls=PPNet,
    fem=fem,
    shell_problem=shell_problem,
    config=cfg,
)

print(cfg)
result = trainer.train(
    shape_path,
    face_tensors=tensors["face_tensors"],
)

TensorBoard log dir: runs/CircularSurf1
TrainingConfig(seed_number=5, use_anisotropy=False, fixed_height=0.2, target_volfrac=0.2, seed_repulsion_sigma=0.08, boundary_margin=0.05, freeze_w=False, gate_sharpen_gamma=2.0, w_min=0.0005, w_max=0.5, lam_fem=0.0, lam_vol=20.0, lam_rep=0.015, lam_bnd=0.0, lam_strut=0.0005, lam_strut_edge=0.0, lam_strut_void=0.2, comp_normalize_by=None, normalize_losses=False, fem_density_floor=0.02, skip_bad_fem_steps=True, num_steps=5000, context_vector_size=8, tau=0.1, beta=0.02, lr_seed_refine=0.003, lr_delta_head=0.0001, lr_mlp=0.0002, lr_w_head=0.002, lr_h_head=0.0002, log_every=100, early_stop_start=500, patience=500, min_delta=1e-10, use_gating=False, lr_gate_head=5e-05, gate_active_threshold=0.3, gate_eps=1e-08, gate_bias_init=2.0, lam_gate_count=0.005, gate_target_count=10.0, lam_gate_binary=0.1, gate_binary_warmup_steps=80, gate_warmup_steps=20, eps=1e-12, use_boundary_weighted_volume=True, boundary_vol_weight=0.2, Offset_scale=5, scheduler_milestone

Training:   0%|          | 0/5000 [00:00<?, ?it/s]

New best_step=0 | best_score=0.138641 | vol_eff=0.116741 | comp=0.000000e+00 | w=2.514524e-01
[00000] | active(total/mean)=0/0.00 | gate(min/mean/max)=0.000/0.000/0.000 | L_total=1.3864e-01 | L_vol=6.932e-03 L_fem=0.000e+00 L_strut=2.299e-05 L_rep=1.579e-12 L_bnd=6.584e-01 L_gate=0.000e+00 L_gbin=0.000e+00 | vol=0.117 vol_eff=0.117 (/0.200) comp=0.000e+00 w=2.515e-01 | Lse=8.406e-01 Lsv=1.149e-04 rho(min/mean/max)=0.000/0.114/0.900 | rho_b=0.000 rho_v=0.114 | Δrho=0.00e+00 Δseed=0.00e+00 grad_mean=1.92e-03 | fem=OK | best=1.3864e-01@0
[00100] | active(total/mean)=0/0.00 | gate(min/mean/max)=0.000/0.000/0.000 | L_total=9.3275e-02 | L_vol=4.664e-03 L_fem=0.000e+00 L_strut=2.713e-09 L_rep=1.449e-06 L_bnd=6.025e-01 L_gate=0.000e+00 L_gbin=0.000e+00 | vol=0.132 vol_eff=0.132 (/0.200) comp=0.000e+00 w=2.906e-01 | Lse=8.207e-01 Lsv=1.357e-08 rho(min/mean/max)=0.000/0.129/0.874 | rho_b=0.000 rho_v=0.129 | Δrho=1.44e-01 Δseed=5.33e-02 grad_mean=2.37e-03 | fem=OK | best=8.9901e-02@96
[00200] | a

In [ ]:
trainer.visualize_best_seed_activity(result, points_xyz, faces_ijk)
print(result["tensorboard_log_dir"])

In [ ]:
smooth = trainer.visualize_result_final_smooth_points(
    result=result,
    shape_or_path=str(shape_path),
    thr=0.3,
    grid_res_u=300,
    grid_res_v=300,
)

In [ ]:
trainer.visualize_result_final_smooth_surface_pyvista(
    result=result,
    shape_or_path=str(shape_path),
    thr=0.3,
    grid_res_u=150,
    grid_res_v=150,
)

Widget(value='<iframe src="http://localhost:35397/index.html?ui=P_0x78c034515900_2&reconnect=auto" class="pyvi…

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
for t in [0.3,0.45,0.5, 0.6, 0.7]:
    trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=t, show_solid=True)